In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "workspace"
schema = "de_ecommerce"

def silver(name):
    return spark.table(f"{catalog}.{schema}.silver_{name}")

def gold(name):
    return spark.table(f"{catalog}.{schema}.gold_{name}")

def save_gold(df, name):
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.gold_{name}")
    print(f"gold_{name}: {gold(name).count()} rows")

In [0]:
geo = silver("geolocation").select("zip_code_prefix", "lat", "lng")

# ---- dim_customer ----
dim_customer = (silver("customers")
    .join(geo.withColumnRenamed("zip_code_prefix", "customer_zip_code_prefix"),
          "customer_zip_code_prefix", "left")
    .withColumn("customer_key", F.row_number().over(Window.orderBy("customer_id")))
    .select("customer_key", "customer_id", "customer_unique_id", "customer_zip_code_prefix",
            "customer_city", "customer_state",
            F.col("lat").alias("customer_lat"), F.col("lng").alias("customer_lng")))
save_gold(dim_customer, "dim_customer")

# ---- dim_seller ----
dim_seller = (silver("sellers")
    .join(geo.withColumnRenamed("zip_code_prefix", "seller_zip_code_prefix"),
          "seller_zip_code_prefix", "left")
    .withColumn("seller_key", F.row_number().over(Window.orderBy("seller_id")))
    .select("seller_key", "seller_id", "seller_zip_code_prefix", "seller_city", "seller_state",
            F.col("lat").alias("seller_lat"), F.col("lng").alias("seller_lng")))
save_gold(dim_seller, "dim_seller")

# ---- dim_product ----
dim_product = (silver("products")
    .withColumn("product_key", F.row_number().over(Window.orderBy("product_id")))
    .select("product_key", "product_id",
            F.col("product_category_name_english").alias("category"),
            F.col("product_category_name").alias("category_pt"),
            "product_name_length", "product_description_length", "product_photos_qty",
            "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"))
save_gold(dim_product, "dim_product")

# ---- dim_date ----
dim_date = (spark.sql("SELECT explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) AS date")
    .select(F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
            "date",
            F.year("date").alias("year"),
            F.quarter("date").alias("quarter"),
            F.month("date").alias("month"),
            F.date_format("date", "MMMM").alias("month_name"),
            F.dayofmonth("date").alias("day"),
            F.date_format("date", "EEEE").alias("day_name"),
            F.dayofweek("date").isin(1, 7).alias("is_weekend")))
save_gold(dim_date, "dim_date")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold_dim_customer: 99441 rows
gold_dim_seller: 3095 rows
gold_dim_product: 32951 rows
gold_dim_date: 1096 rows


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
for t in ["dim_customer", "dim_seller", "dim_product", "dim_date"]:
    print(t)
    display(gold(t).limit(5))

dim_customer


customer_key,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng
1,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,06273,osasco,SP,-23.499062689784747,-46.76793857798514
2,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,itapecerica,MG,-20.46975446206617,-45.123242407312524
3,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,nova venecia,ES,-18.706580825893784,-40.406383792282284
4,0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,mendonca,MG,-21.77059918122806,-43.35523022384441
5,000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,04841,sao paulo,SP,-23.738708912922274,-46.68933298902944


dim_seller


seller_key,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng
1,0015a82c2db000af6aaaf3ae2ecb0532,09080,santo andre,SP,-23.640443596560726,-46.541742273007685
2,001cca7ae9ae17fb1caed9dfb1094831,29156,cariacica,ES,-20.27851289140612,-40.411675223412075
3,001e6ad469a905060d959994f1b41e4f,24754,sao goncalo,RJ,-22.872354587681087,-43.027432812835386
4,002100f778ceb8431b7a1020ff7ab48f,14405,franca,SP,-20.52875913617737,-47.411109783856105
5,003554e2dce176b5555353e4f3555ac8,74565,goiania,GO,-16.64057412255124,-49.27648326906269


dim_product


product_key,product_id,category,category_pt,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1,00066f42aeeb9f3007548bb9d3f33c38,perfumery,perfumaria,53,596,6,300,20,16,16
2,00088930e925c41fd95ebfe695fd2655,auto,automotivo,56,752,4,1225,55,10,26
3,0009406fd7479715e4bef61dd91f2462,bed_bath_table,cama_mesa_banho,50,266,2,300,45,15,35
4,000b8f95fcb9e0096488278317764d19,housewares,utilidades_domesticas,25,364,3,550,19,24,12
5,000d9be29b5207b54e86aa1b1ac54872,watches_gifts,relogios_presentes,48,613,4,250,22,11,15


dim_date


date_key,date,year,quarter,month,month_name,day,day_name,is_weekend
20160101,2016-01-01,2016,1,1,January,1,Friday,false
20160102,2016-01-02,2016,1,1,January,2,Saturday,true
20160103,2016-01-03,2016,1,1,January,3,Sunday,true
20160104,2016-01-04,2016,1,1,January,4,Monday,false
20160105,2016-01-05,2016,1,1,January,5,Tuesday,false


In [0]:
orders = silver("orders")
items = silver("order_items")

d_cust = gold("dim_customer").select("customer_key", "customer_id")
d_sell = gold("dim_seller").select("seller_key", "seller_id")
d_prod = gold("dim_product").select("product_key", "product_id")

date_key = lambda c: F.date_format(c, "yyyyMMdd").cast("int")

fact_order_items = (items
    .join(orders.select("order_id", "customer_id", "order_status", "order_purchase_timestamp"), "order_id")
    .join(d_cust, "customer_id", "left")
    .join(d_sell, "seller_id", "left")
    .join(d_prod, "product_id", "left")
    .withColumn("purchase_date_key", date_key("order_purchase_timestamp"))
    .withColumn("item_total", F.round(F.col("price") + F.col("freight_value"), 2))
    .select("order_id", "order_item_id", "customer_key", "seller_key", "product_key",
            "purchase_date_key", "order_status", "price", "freight_value", "item_total"))
save_gold(fact_order_items, "fact_order_items")

gold_fact_order_items: 112650 rows


In [0]:
item_agg = items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.round(F.sum("price"), 2).alias("items_value"),
    F.round(F.sum("freight_value"), 2).alias("freight_value"))

pay_agg = silver("order_payments").groupBy("order_id").agg(
    F.round(F.sum("payment_value"), 2).alias("payment_value"),
    F.max("payment_installments").alias("max_installments"))

rev_agg = silver("order_reviews").groupBy("order_id").agg(
    F.round(F.avg("review_score"), 2).alias("avg_review_score"))

fact_orders = (orders
    .join(d_cust, "customer_id", "left")
    .join(item_agg, "order_id", "left")
    .join(pay_agg, "order_id", "left")
    .join(rev_agg, "order_id", "left")
    .withColumn("purchase_date_key", date_key("order_purchase_timestamp"))
    .withColumn("delivery_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("estimated_delivery_days", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
    .withColumn("delivery_delay_days", F.datediff("order_delivered_customer_date", "order_estimated_delivery_date"))
    .withColumn("is_late", F.col("delivery_delay_days") > 0)
    .select("order_id", "customer_key", "purchase_date_key", "order_status",
            "items_count", "items_value", "freight_value", "payment_value", "max_installments",
            "avg_review_score", "delivery_days", "estimated_delivery_days",
            "delivery_delay_days", "is_late"))
save_gold(fact_orders, "fact_orders")

gold_fact_orders: 99441 rows


In [0]:
fi = gold("fact_order_items")
print("items with a missing key:",
      fi.filter(F.col("customer_key").isNull() | F.col("seller_key").isNull() | F.col("product_key").isNull()).count())
display(gold("fact_order_items").limit(5))
display(gold("fact_orders").limit(5))

items with a missing key: 0


order_id,order_item_id,customer_key,seller_key,product_key,purchase_date_key,order_status,price,freight_value,item_total
2807d0e504d6d4894d41672727bc139f,1,44365,1709,13534,20180203,delivered,9.5,7.78,17.28
ccbabeb0b02433bd0fcbac46e70339f2,1,77753,258,17663,20180219,delivered,27.9,15.1,43.0
e4de6d53ecff736bc68804b0b6e9f635,1,61935,1536,18617,20171016,delivered,179.99,51.13,231.12
cadbb3657dac2dbbd5b84b12e7b78aad,1,57278,11,20219,20180227,delivered,394.9,14.89,409.79
d3d6788577c9592da441752e8a1dd5e3,1,52015,2482,16087,20170919,delivered,58.99,17.66,76.65


order_id,customer_key,purchase_date_key,order_status,items_count,items_value,freight_value,payment_value,max_installments,avg_review_score,delivery_days,estimated_delivery_days,delivery_delay_days,is_late
2807d0e504d6d4894d41672727bc139f,44365,20180203,delivered,1,9.5,7.78,17.28,1,3.0,5,18,-13,false
ccbabeb0b02433bd0fcbac46e70339f2,77753,20180219,delivered,1,27.9,15.1,43.0,1,4.0,18,22,-4,false
e4de6d53ecff736bc68804b0b6e9f635,61935,20171016,delivered,1,179.99,51.13,231.12,1,2.0,23,36,-13,false
cadbb3657dac2dbbd5b84b12e7b78aad,57278,20180227,delivered,1,394.9,14.89,409.79,1,4.0,17,30,-13,false
d3d6788577c9592da441752e8a1dd5e3,52015,20170919,delivered,1,58.99,17.66,76.65,7,5.0,21,24,-3,false


SCD Type 2 code

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "workspace"
schema = "de_ecommerce"
table = f"{catalog}.{schema}.gold_dim_customer"

In [0]:
spark.sql(f"ALTER TABLE {table} ADD COLUMNS (start_date DATE, end_date DATE, is_current BOOLEAN)")
spark.sql(f"UPDATE {table} SET start_date = DATE'2016-01-01', end_date = DATE'9999-12-31', is_current = true")
display(spark.table(table).limit(5))

customer_key,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng,start_date,end_date,is_current
1,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,06273,osasco,SP,-23.499062689784747,-46.76793857798514,2016-01-01,9999-12-31,true
2,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,itapecerica,MG,-20.46975446206617,-45.123242407312524,2016-01-01,9999-12-31,true
3,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,nova venecia,ES,-18.706580825893784,-40.406383792282284,2016-01-01,9999-12-31,true
4,0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,mendonca,MG,-21.77059918122806,-43.35523022384441,2016-01-01,9999-12-31,true
5,000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,04841,sao paulo,SP,-23.738708912922274,-46.68933298902944,2016-01-01,9999-12-31,true
